# Diagnòstic DPR: els vectors dels documents col·lapsen?
Executa amb el kernel acabat de reiniciar.

In [ ]:
from transformers import BertTokenizer
BertTokenizer.from_pretrained('bert-base-uncased')   # el descarrega abans que el projecte activi el mode offline

import torch
from omegaconf import OmegaConf
from src.data_utils.corpus import Corpus
from src.retrievers.dpr import DPR
from src.training.trainer import build_pairs

corpus = Corpus('fiqa', './retrieval_data')
cfg = OmegaConf.create({'model': OmegaConf.load('config/model/encoder.yaml')})
pairs = build_pairs(corpus.val)[:16]
qs, ds = [q for q, _ in pairs], [d for _, d in pairs]

def check(dpr, nom):
    def enc(e, t):
        tok = dpr.tokenizer(t, padding=True, truncation=True, max_length=256, return_tensors='pt')
        return e(tok['input_ids'].to(dpr.device), tok['attention_mask'].to(dpr.device))
    dpr.query_encoder.eval(); dpr.document_encoder.eval()
    with torch.no_grad():
        q, d = enc(dpr.query_encoder, qs), enc(dpr.document_encoder, ds)
    S = q @ d.T
    print(f'{nom}: std docs {d.std(0).mean():.4f} | std queries {q.std(0).mean():.4f} | std per fila de S {S.std(1).mean():.4f}')

check(DPR(cfg, corpus.val), 'inicial ')
cfg.model.query_encoder_weights_path = 'outputs/dpr_weights/query_encoder.pt'
cfg.model.document_encoder_weights_path = 'outputs/dpr_weights/document_encoder.pt'
check(DPR(cfg, corpus.val), 'entrenat')